In [2]:
import pandas as pd
import numpy as np

train = pd.read_csv("../data/train.csv", parse_dates=["date"])
stores = pd.read_csv("../data/stores.csv")
holidays = pd.read_csv("../data/holidays_events.csv", parse_dates=["date"])

df = train.copy()
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day"] = df["date"].dt.day
df["dow"] = df["date"].dt.dayofweek          # 0 = Pazartesi
df["is_weekend"] = (df["dow"] >= 5).astype(int)
df["is_payday"] = ((df["day"] == 15) | df["date"].dt.is_month_end).astype(int)
df["log_sales"] = np.log1p(df["sales"])      # RMSLE için hedefin log hali

print(df.groupby("is_weekend")["sales"].mean())
print(df.groupby("is_payday")["sales"].mean())

is_weekend
0    321.629081
1    448.179995
Name: sales, dtype: float64
is_payday
0    357.442739
1    362.494890
Name: sales, dtype: float64


In [3]:
print(df.groupby(["is_weekend", "is_payday"])["sales"].mean())

is_weekend  is_payday
0           0            320.995312
            1            330.646242
1           0            448.683069
            1            441.121241
Name: sales, dtype: float64


In [4]:
hol = holidays[(holidays["locale"] == "National") & (~holidays["transferred"])]
print(hol["type"].value_counts())
print("aynı güne düşen en çok satır:", hol.groupby("date").size().max())

type
Event         56
Holiday       52
Additional    40
Transfer       8
Bridge         5
Work Day       5
Name: count, dtype: int64
aynı güne düşen en çok satır: 2


In [5]:
hol = holidays[(holidays["locale"] == "National") & (~holidays["transferred"])]

def flag(types):
    s = hol[hol["type"].isin(types)].drop_duplicates("date").set_index("date")
    return pd.Series(1, index=s.index)

hol_days = (pd.DataFrame({
    "is_holiday": flag(["Holiday", "Transfer", "Additional", "Bridge"]),
    "is_event": flag(["Event"]),
    "is_workday_makeup": flag(["Work Day"]),
}).fillna(0).astype(int).rename_axis("date").reset_index())

n_before = len(df)
df = df.merge(hol_days, on="date", how="left")
for c in ["is_holiday", "is_event", "is_workday_makeup"]:
    df[c] = df[c].fillna(0).astype(int)

assert len(df) == n_before      # birleştirme satır çoğaltmadı
print(df.groupby("is_holiday")["sales"].mean())
print(df.groupby("is_event")["sales"].mean())

is_holiday
0    354.537659
1    424.447188
Name: sales, dtype: float64
is_event
0    355.505229
1    425.024432
Name: sales, dtype: float64


In [6]:
print(df.groupby(["is_weekend", "is_holiday"])["sales"].mean())

is_weekend  is_holiday
0           0             316.914864
            1             414.694218
1           0             447.982563
            1             452.730799
Name: sales, dtype: float64


In [7]:
n_before = len(df)
df = df.merge(stores, on="store_nbr", how="left")   # city, state, type, cluster
assert len(df) == n_before

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

cat_cols = ["family", "store_nbr", "type", "city"]
num_cols = ["onpromotion", "cluster"]
bin_cols = ["is_weekend", "is_payday", "is_holiday", "is_event", "is_workday_makeup"]

pre = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", StandardScaler(), num_cols),
], remainder="passthrough")

X = df[cat_cols + num_cols + bin_cols]
X_t = pre.fit_transform(X)
print(X_t.shape, type(X_t))

(3000888, 121) <class 'scipy.sparse._csr.csr_matrix'>
